In [1]:
import sys

sys.path.append('../../pythonside')

import numpy as np
import pandas as pd

from functions import (
    correlation, regression_slope, covariance,
    y_intercept, standard_deviation, mean
)  

from graph import scatter_with_regression 

In [2]:
# Load the datasets, explicitly handling potential file not found errors.
try:
    breast_data = pd.read_csv('../../data/breast-country.csv')
    print("breast-country.csv loaded successfully.")
except FileNotFoundError:
    print("Error: breast-country.csv not found.  Check the file path.")

try:
    gdp_data = pd.read_csv('../../data/API_NY.GDP.MKTP.CD_DS2_en_csv_v2_88.csv', skiprows=4)
    print("GDP data loaded successfully.")
except FileNotFoundError:
    print("Error: GDP data not found. Check the file path.")

breast-country.csv loaded successfully.
GDP data loaded successfully.


In [3]:
def clean_and_merge():
    """
    Cleans and merges GDP and breast cancer data.

    Returns:
        pandas.DataFrame: Merged DataFrame with cleaned data, or None if an error occurs.
    """
    try:
        gdp_data = pd.read_csv('../../data/API_NY.GDP.MKTP.CD_DS2_en_csv_v2_88.csv', skiprows=4)
        breast_data = pd.read_csv('../../data/breast-country.csv')
    except FileNotFoundError as e:
        print(f"Error loading data: {e}. Make sure the data files are in the correct directory.")
        return None

    gdp_clean = gdp_data[['Country Code', '2023']].rename(columns={'2023': 'GDP'})
    breast_clean = breast_data[['Alpha-3 code', 'ASR (World) per 100 000']]

    merged = pd.merge(
        breast_clean,
        gdp_clean,
        left_on='Alpha-3 code',
        right_on='Country Code',
        how='inner'
    )

    merged['GDP'] = pd.to_numeric(merged['GDP'], errors='coerce')
    merged['ASR (World) per 100 000'] = pd.to_numeric(merged['ASR (World) per 100 000'], errors='coerce')
    return merged.dropna()

In [4]:
df = clean_and_merge()

if df is None:
    print("Data cleaning and merging failed.  Check file paths and data integrity.")
else:
    print("Data cleaning and merging successful.")
    print(df.head()) 

KeyError: "['Alpha-3 code'] not in index"

In [ ]:
if df is not None and not df.empty:
    corr_coef = correlation(df['GDP'], df['ASR (World) per 100 000'])
    slope = regression_slope(df['GDP'], df['ASR (World) per 100 000'])
    covar = covariance(df['GDP'], df['ASR (World) per 100 000'])

    print(f"Correlation Coefficient: {corr_coef:.3f}")
    print(f"Regression Slope: {slope:.2e}")
    print(f"Covariance: {covar:.2e}")
else:
    print("Cannot perform statistical analysis. DataFrame is empty or None.")

In [ ]:
if df is not None and not df.empty:
    scatter_with_regression(
        x=df['GDP'],
        y=df['ASR (World) per 100 000'],
        x_label='GDP (current US$)',
        y_label='Age-Standardized Rate per 100k',
        title='Breast Cancer ASR vs GDP (2023)'
    )
else:
    print("Cannot create visualization. DataFrame is empty or None.")